# 🎬 CineMatch AI: Content-Based Movie Recommender System
### End-to-End Machine Learning Pipeline & Exploratory Analysis

This notebook details the full data science and natural language processing (NLP) pipeline used to build the CineMatch AI recommender system based on the TMDB 5000 Movies & Credits dataset.

## 1. Imports and Setup

In [ ]:
import ast
import pickle
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import nltk
from nltk.stem.porter import PorterStemmer

print("Libraries successfully loaded.")

## 2. Dataset Loading & Merging
We merge `tmdb_5000_movies.csv` and `tmdb_5000_credits.csv` on the shared `title` column.

In [ ]:
# Load raw datasets
# movies = pd.read_csv('tmdb_5000_movies.csv')
# credits = pd.read_csv('tmdb_5000_credits.csv')
# df = movies.merge(credits, on='title')
# df.head(2)

## 3. Metadata Extraction & Feature Engineering
We parse JSON strings in `genres`, `keywords`, `cast`, and `crew` to extract:
- Genre labels
- Plot keywords
- Top 3 lead actors
- Film Director

In [ ]:
def parse_json(text):
    return [i['name'] for i in ast.literal_eval(text)] if isinstance(text, str) else []

def get_top_cast(text, limit=3):
    return [i['name'] for i in ast.literal_eval(text)[:limit]] if isinstance(text, str) else []

def get_director(text):
    if not isinstance(text, str):
        return []
    for i in ast.literal_eval(text):
        if i.get('job') == 'Director':
            return [i['name']]
    return []

def collapse(items):
    return [x.replace(' ', '') for x in items]

## 4. NLP Tokenization, Stemming & Vectorization
We collapse spaces in entity names (e.g. `'James Cameron'` -> `'JamesCameron'`), combine all descriptive tokens into a unified `tags` document, apply Porter Stemming, and compute a 5,000-dimensional bag-of-words representation.

In [ ]:
ps = PorterStemmer()

def stem(text):
    return ' '.join([ps.stem(w) for w in text.split()])

# Example:
print("Stemming test:", stem("loving loves loved action actions"))

## 5. Pairwise Cosine Similarity & Vector Space Model
Cosine similarity measures the cosine of the angle between two multi-dimensional feature vectors:

$$\text{similarity}(\vec{u}, \vec{v}) = \frac{\vec{u} \cdot \vec{v}}{\|\vec{u}\| \|\vec{v}\|}$$

Values range from 0 (completely orthogonal/dissimilar) to 1 (identical orientation in tag space).

In [ ]:
# cv = CountVectorizer(max_features=5000, stop_words='english')
# vectors = cv.fit_transform(final_df['tags']).toarray()
# similarity = cosine_similarity(vectors)
# print('Similarity matrix shape:', similarity.shape)

## 6. Recommendation Verification

In [ ]:
with open('movie_dict.pkl', 'rb') as f:
    movies = pd.DataFrame(pickle.load(f))
with open('similarity.pkl', 'rb') as f:
    similarity = pickle.load(f)

def test_recommend(movie_title, top_n=5):
    idx = movies[movies['title'] == movie_title].index[0]
    distances = similarity[idx]
    ranked = sorted(list(enumerate(distances)), reverse=True, key=lambda x: x[1])[1:top_n+1]
    for rank, (i, score) in enumerate(ranked, 1):
        print(f"{rank}. {movies.iloc[i].title} - {round(score * 100, 1)}% match")

test_recommend('The Dark Knight')